# 作用域、闭包与高阶函数

学习目标：理解函数如何查找和保留名称，并用函数对象组织可复用的处理规则。

前置知识：名称绑定、可变对象与引用、列表和字典、条件与循环、推导式、函数参数与返回值、默认值参数。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

## 1 命名空间与作用域

命名空间（namespace）保存名称到对象的对应关系。函数每次调用都有自己的局部命名空间，因此两个函数可以各自使用同一个名称。

作用域（scope）是代码中能直接使用某个名称的范围。命名空间回答“名称对应什么对象”，作用域回答“在哪里能直接找到它”；两者描述的是同一套名称绑定的不同方面。

In [1]:
topic = "公共主题"


def read_python_topic():
    """返回本函数使用的主题。"""
    topic = "Python"
    return topic


def read_sql_topic():
    """返回另一个函数使用的主题。"""
    topic = "SQL"
    return topic


print(read_python_topic(), read_sql_topic())  # Python SQL
print(topic)  # 公共主题；两个局部赋值没有改动外部绑定。

Python SQL
公共主题


## 2 普通函数的名称查找

### 2.1 按 LEGB 理解查找顺序

对本章这类普通函数，未被 global 或 nonlocal 改变归属的名称，按下面的范围理解查找顺序。LEGB 是四个英文名称的首字母缩写；没有外层函数时，跳过 E。

| 原文名称 | 中文名称／含义 |
| --- | --- |
| Local | 局部：当前函数的参数和局部名称 |
| Enclosing | 外层函数：按定义位置，由近到远查找包围它的函数 |
| Global | 全局：函数定义所在模块的名称 |
| Builtins | 内置：len、print 等解释器提供的名称 |

模块是组织 Python 代码的单位，完整导入用法在“模块与包”中学习。本 Notebook 同一内核中的顶层定义共享全局名称，后续示例会就地设置需要的值。

In [2]:
unit = "分"


def describe_score():
    """组合来自局部、外层函数、全局和内置范围的名称。"""
    course = "Python"

    def format_score(score):
        """将一个分数格式化为说明文字。"""
        return f"{course}: {score}{unit}，名称长度 {len(course)}"

    return format_score(90)


# score 来自 L，course 来自 E，unit 来自 G，len 来自 B。
print(describe_score())  # Python: 90分，名称长度 6

Python: 90分，名称长度 6


### 2.2 定义位置决定可查找的范围

外层函数指代码中包围定义的函数，不是运行时调用它的函数。函数被谁调用，不会改变它所属的全局命名空间；全局名称的当前值仍在调用时读取。

较近范围中的同名绑定会遮蔽外部名称。因此不要随意用 len、list 等内置名称命名变量，否则后续调用可能找到自己的对象。

In [3]:
message = "定义所在的全局范围"


def read_message():
    """读取定义所在范围中的 message。"""
    return message


def call_reader():
    """在不同局部范围中调用读取函数。"""
    message = "调用者的局部范围"
    return read_message(), message


print(call_reader())
# ('定义所在的全局范围', '调用者的局部范围')；不会沿调用者查找。
message = "更新后的全局内容"
print(read_message())  # 更新后的全局内容；函数没有冻结旧值。

('定义所在的全局范围', '调用者的局部范围')
更新后的全局内容


### 2.3 缩进不一定创建新作用域

普通 if、for 语句不会创建单独的局部作用域；其中绑定的名称仍属于所在函数或模块。列表推导式的循环变量则与外层隔离。

LEGB 也不是所有查找行为的总规则：点号后的属性名属于属性查找；类体的名称不会自动成为方法的外层函数变量。类与属性的规则在后续类相关主题展开。

Python 3.12 的 type 语句和类型形参还引入标注作用域（annotation scope），不能一概套用普通函数规则；其用途在类型标注与泛型主题中说明。

In [4]:
def compare_loop_names():
    """比较普通语句和推导式的名称归属。"""
    if True:
        label = "分支中绑定的名称"
    for number in [2, 4]:
        last = number
    squares = [number * number for number in [1, 3]]
    return label, last, number, squares


# if 和普通 for 的名称仍可读；推导式没有把 number 改成 3。
print(compare_loop_names())
# ('分支中绑定的名称', 4, 4, [1, 9])

('分支中绑定的名称', 4, 4, [1, 9])


## 3 赋值如何影响名称归属

### 3.1 为什么会出现 UnboundLocalError

函数内一旦有给某名称赋值的操作，且未声明 global 或 nonlocal，该名称就会被判定为局部名称；判断依据是整个函数体，不是代码是否已经执行到赋值处。

下面的 count += 1 需要先读取 count，再重新绑定它。局部 count 尚无值时会引发 UnboundLocalError，不能因为全局有同名值就退回全局查找。

这里临时使用 try 执行预期失败的调用，except UnboundLocalError 只处理这一种错误，as error 保存错误对象供打印。完整异常处理在“异常处理与调试”中学习；捕获错误只是让教学继续，不能修复名称归属。

In [5]:
count = 10


def increment_bad():
    """演示局部名称尚未绑定时执行增强赋值的错误。"""
    count += 1
    return count

In [6]:
# 预期 UnboundLocalError：直接观察原始异常，之后继续运行下一单元。
increment_bad()

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value

In [7]:
print(count)  # 10：错误发生后，全局 count 没有被更新。
# 错误信息指出局部 count 尚未关联值，而不是说全局 count 不存在。

10


### 3.2 global 与显式传参

global 声明让函数中的指定名称指向模块级绑定；声明应放在该函数对名称的读取或赋值之前。只读取全局名称不需要声明，重新绑定它才需要。

global 不表示“所有模块共同拥有”。下例只演示它对当前全局 count 的作用。日常代码优先通过参数接收数据、通过返回值交付结果，减少对共享状态的隐式修改。

In [8]:
count = 10


def increment_global():
    """教学演示：更新当前模块中的 count。"""
    global count
    count += 1
    return count


def increment(value):
    """返回增加后的整数，不修改调用者的名称绑定。"""
    return value + 1


print(increment_global(), count)  # 11 11；全局绑定发生变化。
original = 10
updated = increment(original)
print(original, updated)  # 10 11；调用者自行决定是否采用返回值。

11 11
10 11


### 3.3 nonlocal 选择最近的外层函数绑定

nonlocal 用于重新绑定外层函数已经绑定的名称，不会跳到全局。若多个外层函数都有同名绑定，选择最近的一层。

声明应在当前函数使用该名称之前。外层函数没有相应绑定时，nonlocal 会产生 SyntaxError；它不能替外层函数创建一个原本不存在的名称。

In [9]:
def compare_nonlocal():
    """观察 nonlocal 只修改最近的外层同名绑定。"""
    label = "最外层"

    def middle():
        """在中间层建立同名绑定。"""
        label = "中间层"

        def rename():
            """更新最近一层函数中的 label。"""
            nonlocal label
            label = "中间层已修改"

        rename()
        return label

    return middle(), label


print(compare_nonlocal())  # ('中间层已修改', '最外层')

('中间层已修改', '最外层')


### 3.4 修改对象不等于重新绑定名称

对外部列表调用 append，修改的是已经找到的列表对象；给名称重新赋值，才改变名称到对象的绑定。因此，修改外部列表内容本身不需要 nonlocal。

这与容器章节的共享引用是同一个现象：作用域决定找到哪一个名称，对象的可变性决定能否修改它的内容。

In [10]:
def compare_list_changes():
    """比较修改外层列表与创建同名局部列表。"""
    notes = ["开始"]

    def append_note():
        """修改外层名称所指向的列表。"""
        notes.append("继续")

    def replace_note():
        """创建自己的局部绑定。"""
        notes = ["仅在内部"]
        return notes

    append_note()
    local_notes = replace_note()
    return notes, local_notes


print(compare_list_changes())
# (['开始', '继续'], ['仅在内部'])；局部重新绑定不替换外层列表。

(['开始', '继续'], ['仅在内部'])


## 4 把函数当作对象使用

### 4.1 函数名称、别名与调用结果

执行 def 会创建函数对象并绑定名称，函数体在调用时才运行。函数可以赋给其他名称，也可以存入列表或字典。

下面 greet 是函数对象的名称，greet("小林") 是一次调用，其值是返回的字符串。别名保存同一个函数对象；这也是对象引用规则在函数上的应用。

In [11]:
def greet(name):
    """返回给指定姓名的问候。"""
    return f"你好，{name}"


alias = greet
actions = {"welcome": greet}
text = greet("小林")

print(alias is greet)  # True；赋值没有复制函数。
print(actions["welcome"] is greet)  # True；字典也能保存函数引用。
print(alias("小周"))  # 你好，小周；通过别名调用。
print(text)  # 你好，小林；text 保存返回值，不是函数。

True
True
你好，小周
你好，小林


### 4.2 让调用者提供处理函数

高阶函数（higher-order function）指接收函数作为参数，或返回函数的函数。把函数传进去，可以让遍历流程复用，而具体处理规则由调用者决定。

回调函数（callback）是交给另一段代码、由它在需要时调用的函数。高阶函数描述接口能力，回调描述被传入函数承担的角色；下面的回调就在普通循环中立即执行，不涉及异步机制。

In [12]:
def transform_names(names, transform):
    """对每个姓名调用 transform，并收集返回值。"""
    return [transform(name) for name in names]


def strip_name(name):
    """去除姓名两端的空白。"""
    return name.strip()


names = [" 小林 ", " 小周"]
print(transform_names(names, strip_name))  # ['小林', '小周']
print(names)  # [' 小林 ', ' 小周']；结果写入新列表。
# 传 strip_name 是交出函数；写成 strip_name(...) 会先调用它。

['小林', '小周']
[' 小林 ', ' 小周']


### 4.3 lambda 适合短表达式

lambda name: name.strip() 创建一个匿名函数；name 是它的形参，这里接收一个姓名字符串，冒号右侧的表达式就是返回值。lambda 只能包含一个表达式，不能放入普通赋值语句或多条语句；复杂处理使用 def 更易阅读。

lambda 是创建函数的写法，高阶函数描述接收或返回函数的行为，两者不是同一分类。下面沿用上一例的 transform\_names，将一个短 lambda 作为回调。

In [13]:
names = [" 小林 ", " 小周"]

print(transform_names(names, lambda name: name.strip()))
# ['小林', '小周']；与前面的具名函数承担相同的处理工作。


def format_name(name):
    """清理空白并为姓名添加称呼。"""
    cleaned = name.strip()
    return f"{cleaned}同学"


print(transform_names(names, format_name))  # ['小林同学', '小周同学']
# 需要多步说明时，具名函数能保留清楚的局部名称和文档字符串。

['小林', '小周']
['小林同学', '小周同学']


## 5 闭包保留外层函数的绑定

### 5.1 返回一个仍能使用外层变量的函数

当内部函数使用外层函数的局部变量时，它可以保留对这些绑定的访问，即使外层调用已经结束。函数连同这类保留的绑定构成闭包（closure）。

本例中 offset 是外层函数的整数偏移量，value 是之后调用内部函数时传入的整数。对内部函数而言，offset 不在本函数内绑定，是来自外层函数的自由变量（free variable）。

嵌套函数说明定义位置；只有使用外层函数变量的绑定时，才涉及这里的闭包机制，不能把所有嵌套函数都当作闭包。

In [14]:
def make_adder(offset):
    """创建一个给整数加上指定偏移量的函数。"""
    def add(value):
        """使用创建时所属的外层绑定处理整数。"""
        return value + offset

    return add


add_two = make_adder(2)
add_ten = make_adder(10)
# make_adder 已经返回，但两个内部函数仍能使用各自的 offset。
print(add_two(5), add_ten(5))  # 7 15
print(add_two is add_ten)  # False；两次调用创建了不同的函数对象。
# return add 返回函数；return add(...) 会先调用并返回计算结果。

7 15
False


### 5.2 用 nonlocal 更新闭包中的状态

闭包保留的是绑定，不会自动冻结变量的旧值。同一次外层调用创建的多个内部函数，可以共享同一个外层绑定；其中一个函数更新它，另一个就能读到新值。

下面的工厂函数每次创建一组计数操作。计数只属于这次创建过程，无需使用全局 count；不同创建过程的整数计数互不影响。

In [15]:
def make_counter():
    """返回共享一次计数状态的增加函数和读取函数。"""
    count = 0

    def advance():
        """增加这组操作持有的计数。"""
        nonlocal count
        count += 1
        return count

    def current():
        """读取这组操作持有的当前计数。"""
        return count

    return advance, current


advance_a, current_a = make_counter()
advance_b, current_b = make_counter()
print(current_a(), current_b())  # 0 0；两次创建各自初始化。
print(advance_a(), advance_a(), current_a())  # 1 2 2；共享最新绑定。
print(advance_b(), current_b(), current_a())  # 1 1 2；两组互不影响。

0 0
1 2 2
1 1 2


## 6 循环闭包与延迟绑定

### 6.1 为什么多个函数读到同一个值

循环中定义函数，不会为每轮循环自动创建独立的外层函数作用域。下面返回的函数共享同一次外层调用中的 offset，等循环结束再调用时，读取的是那时的绑定值，这通常称为延迟绑定（late binding）。

该行为既适用于 lambda，也适用于 def。若循环写在模块顶层，函数读取的是全局循环名称；若像下面这样写在外层函数里，使用的就是闭包绑定。两种写法都不能把“定义时的值”当成自动保存的副本。

In [16]:
def make_late_adders():
    """演示同一次循环中的函数共享外层 offset。"""
    named = []
    anonymous = []
    for offset in range(3):
        def add(value):
            """在调用时读取共享的 offset。"""
            return value + offset

        named.append(add)
        anonymous.append(lambda value: value + offset)
    return named, anonymous


named, anonymous = make_late_adders()
print([add(10) for add in named])  # [12, 12, 12]
print([add(10) for add in anonymous])  # [12, 12, 12]
# 循环结束时 offset 为 2；不同函数对象可以共享同一个外层绑定。
print(named[0] is named[1])  # False；不是列表重复存了同一个函数。

[12, 12, 12]
[12, 12, 12]
False


### 6.2 用默认值保存本轮对象

默认值在执行函数定义时求值，而函数体在调用时执行。下面 saved=offset 把本轮的整数对象作为 saved 的默认值；saved 是内部函数的局部参数。

这种方法利用的是默认值规则。它不会深拷贝对象，若保存的是可变对象，后续对该对象内容的修改仍然可见；本例用不可变整数。

In [17]:
def make_default_adders():
    """用默认值保存每轮循环的整数偏移量。"""
    adders = []
    for offset in range(3):
        adders.append(lambda value, saved=offset: value + saved)
    return adders


adders = make_default_adders()
print([add(10) for add in adders])  # [10, 11, 12]
print(adders[0](10, saved=8))  # 18；saved 仍是可以显式传入的参数。
# 用 def add(value, saved=offset) 定义也遵循相同的默认值规则。

[10, 11, 12]
18


### 6.3 用工厂调用分开绑定

另一种办法是让每轮循环调用一次工厂函数。每次工厂调用有自己的参数绑定，返回的内部函数因此不会共享同一个循环变量。

这里沿用前面的 make\_adder，每次调用分别绑定 offset。返回的 add 只接收 value，调用者不需要知道工厂保存偏移量的方式。

In [18]:
# 使用第 5 节已经定义的 make_adder。
def make_factory_adders():
    """通过独立工厂调用创建三个加法函数。"""
    return [make_adder(offset) for offset in range(3)]


adders = make_factory_adders()
print([add(10) for add in adders])  # [10, 11, 12]
print(adders[0](20), adders[2](20))  # 20 22；换输入后规则仍各自保留。
# 默认值方案把偏移量放在形参默认值中；工厂方案把它留在闭包中。

[10, 11, 12]
20 22


## 7 递归与终止条件

### 7.1 让调用逐步接近基例

递归（recursion）是在求解过程中再次调用自身，把当前问题交给规模更小的同类问题。必须明确基例（base case），也就是无需继续递归就能直接得到结果的情况，并保证每一步都向它靠近。

下面只接收较小的非负整数 n，计算它的阶乘：n 为 0 时返回 1，否则将 n 乘以 n − 1 的阶乘。每次调用都有自己的局部参数 n，内部调用返回后，外层调用才能完成乘法。

递归描述函数的调用方式，与高阶函数接收或返回函数的分类不同。

In [19]:
def factorial(n):
    """计算较小非负整数 n 的阶乘。"""
    if n == 0:
        return 1
    return n * factorial(n - 1)


print(factorial(0), factorial(1), factorial(4))  # 1 1 24
# factorial(4) 依次调用参数为 3、2、1、0 的自身。
# 基例返回 1 后，逐层得到 1、2、6、24；每层都要返回计算结果。
# 不调用负数：负数不断减 1 无法到达这里设置的基例。

1 1 24


### 7.2 递归深度不是累计调用次数

递归深度关注尚未返回的嵌套调用层数。即使有正确基例，过大的输入也可能在到达基例之前触发 RecursionError，表示超过了解释器允许的递归深度。

sys.getrecursionlimit() 可读取当前限制；不要为了绕过错误随意提高 sys.setrecursionlimit() 设置的上限，过高的值可能导致解释器崩溃。这里不改限制，也不故意耗尽调用栈。

如果任务能用简单循环表达，可以避免为每一步增加递归调用。下面与上一例的 factorial 比较相同的小输入，仅检查结果一致，不据此作性能结论。

In [20]:
def factorial_loop(n):
    """用循环计算非负整数 n 的阶乘。"""
    product = 1
    for factor in range(1, n + 1):
        product *= factor
    return product


# 依赖上一例的 factorial；覆盖基例、一步递归和普通小输入。
for n in [0, 1, 5]:
    recursive = factorial(n)
    iterative = factorial_loop(n)
    print(n, recursive, iterative, recursive == iterative)
# 依次为：0 1 1 True、1 1 1 True、5 120 120 True。
# 循环的 n=0 情况没有乘法步骤，初始值 1 就是所需结果。

0 1 1 True
1 1 1 True
5 120 120 True


## 8 map 与 filter 返回迭代器

### 8.1 map 逐项转换

map 接收处理函数和可迭代对象，把每个元素交给函数处理。可迭代对象是能依次提供元素的对象，例如列表；map 返回的迭代器（iterator）则是逐项提供结果、记录消费进度的对象。

创建 map 时不会先算好全部结果；下面用 list 消费迭代器，收集剩余结果。耗尽后再次消费同一个迭代器，不会从头计算。“迭代器与生成器”将进一步介绍完整协议。

下表中的 function 是处理函数，iterable 是提供输入元素的对象。这两个 API 都是高阶函数，其返回值同属迭代器，但承担的处理不同。

| 名称 | 中文名称／含义 | 本章用法 |
| --- | --- | --- |
| map | 映射：产生函数的返回值 | map(function, iterable) |
| filter | 筛选：保留满足条件的原元素 | filter(function, iterable) |

In [21]:
def parse_score(text):
    """把分数字符串转成整数，并显示实际处理时机。"""
    print("转换", text)  # 消费 map 时依次打印“转换 3”“转换 7”。
    return int(text)


texts = ["3", "7"]
scores = map(parse_score, texts)
print("map 已创建")  # 先出现这一行，此时尚未调用 parse_score。
print(list(scores))  # 接着输出“转换 3”“转换 7”和 [3, 7]。
print(list(scores))  # []；同一个迭代器已经耗尽。
print([int(text) for text in texts])  # [3, 7]；直接构造列表的替代写法。
# map 也可接收多个输入，其函数须接收对应数量的实参，
# 并在最短输入耗尽时停止；本章先使用单个输入。

map 已创建
转换 3
转换 7
[3, 7]
[]
[3, 7]


### 8.2 filter 保留原元素

filter 用函数返回值的真值决定是否保留输入元素，返回的不是那些判断结果。用于判断条件的函数常称为谓词（predicate）；下面的谓词接收一个整数，判断它是否为正数。

对简单的转换或筛选，列表推导式通常能直接表达意图；如果需要保留逐项消费的方式，可以使用 map 或 filter。不要把列表推导式得到的列表与迭代器当作同一种结果对象。

In [22]:
def is_positive(value):
    """判断整数是否为正数。"""
    return value > 0


values = [-2, 0, 4, 7]
selected = filter(is_positive, values)
print(list(selected))  # [4, 7]；保留原整数，不是 [True, True]。
print(list(selected))  # []；filter 的结果也会耗尽。
print([value for value in values if value > 0])  # [4, 7]
print(list(map(is_positive, values)))  # [False, False, True, True]
# 对同一判断函数，map 收集返回值，filter 用真值筛选输入。
print(list(filter(None, [0, 1, "", "A"])))  # [1, 'A']；None 表示按自身真值筛选。

[4, 7]
[]
[4, 7]
[False, False, True, True]
[1, 'A']


## 9 用 key 提供排序规则

### 9.1 键函数决定比较什么

sorted 接收可迭代对象，返回新的有序列表。key 接收一个函数，对每个元素计算用于比较的键；它不是接收两个元素的比较函数。

排序键（key）是用来比较的值，与字典键是不同用途。这里 key=get\_minutes 传入函数本身；get\_minutes 的参数是单条任务记录，返回其中的整数分钟数。

一次排序对每个输入元素调用键函数一次。键相等时保留这些元素原有的相对次序，称为稳定排序（stable sort）。

In [23]:
def get_minutes(task):
    """返回任务预计耗时，并显示键函数的调用。"""
    print("取键", task["name"])  # 按输入次序打印：取键 整理、取键 阅读、取键 练习。
    return task["minutes"]


tasks = [
    {"name": "整理", "minutes": 20},
    {"name": "阅读", "minutes": 10},
    {"name": "练习", "minutes": 20},
]
ordered = sorted(tasks, key=get_minutes)
# 三条“取键”输出各对应一条输入记录；返回列表中仍然是任务记录。
print([task["name"] for task in ordered])  # ['阅读', '整理', '练习']
# 整理和练习同为 20 分钟，保持原先的先后次序。
print([task["name"] for task in tasks])  # ['整理', '阅读', '练习']
# sorted 创建新列表，没有原地重排 tasks。

取键 整理
取键 阅读
取键 练习
['阅读', '整理', '练习']
['整理', '阅读', '练习']


### 9.2 多个排序条件用元组表示

元组按字典序比较：先比较第一项，若相等，再比较第二项。键函数可以返回元组，把主次条件写在一个位置。

下面每条记录是“任务名、整数分钟数”二元组，task 表示其中一条记录。键 (-task[1], task[0]) 用分钟数的相反数实现耗时降序，再用名称升序打破平局。

reverse=True 会反转整个键的排序方向；只想让其中一个数值条件降序时，不应把它当作单字段开关。

In [24]:
tasks = [("beta", 20), ("alpha", 20), ("gamma", 30)]

ordered = sorted(tasks, key=lambda task: (-task[1], task[0]))
print(ordered)  # [('gamma', 30), ('alpha', 20), ('beta', 20)]
# 相反数中 -30 小于 -20，所以较长任务在前；平局时 alpha 在 beta 前。

reversed_keys = sorted(
    tasks,
    key=lambda task: (task[1], task[0]),
    reverse=True,
)
print(reversed_keys)  # [('gamma', 30), ('beta', 20), ('alpha', 20)]
# 此时整个元组按反向排序，平局时名称也变成降序。

[('gamma', 30), ('alpha', 20), ('beta', 20)]
[('gamma', 30), ('beta', 20), ('alpha', 20)]


## 本章小结

（1）命名空间保存名称绑定，作用域决定哪里能直接查找；普通函数按定义位置理解 LEGB，局部名称未绑定时不能靠全局同名值补救。

（2）global 与 nonlocal 改变名称的绑定归属，修改可变对象的内容是另一回事。优先显式传参，必要时才使用共享状态。

（3）函数能被保存、传入和返回。lambda 是短函数的写法，闭包保留外层绑定；循环中的默认值与工厂调用可以分别解决延迟绑定问题。

（4）递归要有可到达的基例，也受深度限制。map 和 filter 返回可耗尽的迭代器；sorted 的 key 返回比较依据，元组能表达主次条件。

自查：能否指出一个名称属于哪个范围、它何时被读取，以及传递出去的是函数本身还是一次调用的结果？

## 练习

### 练习 1：预测闭包的读取结果

先隐藏输出，写出两个 print 的结果，再运行核对。说明 read 查找的 level 属于哪个范围，并解释为什么修改顶层 level 与修改外层函数内的 level 作用不同。

可验证标准：两行预测都与运行一致，且能指出 read 读取的是哪一次外层调用中的绑定。

In [25]:
level = 1


def make_reader():
    """创建一个读取外层 level 的函数。"""
    level = 2

    def read():
        """返回所保留绑定的当前值。"""
        return level

    level = 3
    return read


reader = make_reader()
level = 4
print(reader())
print(level)
# 先写下预测与查找路径，再展开输出核对。

3


4


### 练习 2：实现独立的累计器

完成 make\_total，让它接收初始整数 start，返回一个接收整数 delta 的函数；每次调用都把 delta 加到累计值上，并返回更新后的值。要求使用 nonlocal，不修改全局名称。

可验证标准：初始值为 10 的累计器依次加 3、加 −2、加 0，应得到 13、11、11；另一个从 0 开始、只加 5 的累计器得到 5，两者互不影响。

下面用 None 标记尚未完成；补好函数后，运行已有调用检查结果。

In [26]:
def make_total(start=0):
    """练习：创建从 start 开始、接收整数增量的累计器。"""
    # 在这里定义内部函数并返回它；完成后移除这个占位返回。
    return None


total_a = make_total(10)
total_b = make_total(0)
if total_a is None or total_b is None:
    print("待完成：make_total")  # 占位函数尚未完成时只显示这条提示。
else:
    observed = [total_a(3), total_a(-2), total_b(5), total_a(0)]
    print(observed)  # 完成后与题目给出的四次累计要求逐项核对。
    print("符合任务要求：", observed == [13, 11, 5, 11])  # 正确实现时为 True。

待完成：make_total


### 练习 3：组合筛选与排序回调

完成 select\_names(records, keep, key)：records 是由任务字典组成的列表，keep 接收单条记录并判断是否保留，key 接收单条记录并返回排序键。函数应先筛选，再排序，最后返回任务名列表，不改动输入列表。

下面的调用要求保留至少 20 分钟的任务，按耗时降序、名称升序排列。可以用简单推导式或 filter 筛选，不必强行同时使用所有工具。

可验证标准：得到 ["gamma", "alpha", "delta"]；原列表名称顺序仍为 ["alpha", "beta", "gamma", "delta"]，空输入返回空列表。

In [27]:
def select_names(records, keep, key):
    """练习：按调用者提供的条件和排序规则返回任务名。"""
    # 在这里完成筛选、排序与名称提取。
    return None


tasks = [
    {"name": "alpha", "minutes": 20},
    {"name": "beta", "minutes": 10},
    {"name": "gamma", "minutes": 30},
    {"name": "delta", "minutes": 20},
]
# 下面只准备调用与核对条件；先完成函数，再运行这组练习。
chosen = select_names(
    tasks,
    lambda task: task["minutes"] >= 20,
    lambda task: (-task["minutes"], task["name"]),
)
# 未完成时只显示提示，避免占位返回值阻断后续练习。
if chosen is None:
    print("待完成：select_names")  # 占位函数尚未完成时只显示这条提示。
else:
    print(chosen)  # 核对筛选后的名单及排序规则。
    print("排序正确：", chosen == ["gamma", "alpha", "delta"])  # 正确实现时为 True。
    original_names = [task["name"] for task in tasks]
    print("输入顺序保留：", original_names == ["alpha", "beta", "gamma", "delta"])  # 未改动输入时为 True。
    empty = select_names([], lambda task: True, lambda task: task["minutes"])
    print("空输入正确：", empty == [])  # 正确实现时为 True。

待完成：select_names

## 参考与引用来源

| 网站及版本 | 支持的主题与直接定位 |
| --- | --- |
| docs.python.org · Python 3.12 官方文档 | 命名空间、作用域与定义位置：[教程 §9.2](https://docs.python.org/zh-cn/3.12/tutorial/classes.html#python-scopes-and-namespaces)；局部判定、自由变量与类边界：[执行模型 §4.2.1–4.2.2](https://docs.python.org/zh-cn/3.12/reference/executionmodel.html#binding-of-names)，标注作用域：[§4.2.3](https://docs.python.org/zh-cn/3.12/reference/executionmodel.html#annotation-scopes)；绑定声明：[global](https://docs.python.org/zh-cn/3.12/reference/simple_stmts.html#the-global-statement)、[nonlocal](https://docs.python.org/zh-cn/3.12/reference/simple_stmts.html#the-nonlocal-statement)；函数别名与参数引用：[教程 §4.8](https://docs.python.org/zh-cn/3.12/tutorial/controlflow.html#defining-functions)，默认值求值时机：[§4.9.1](https://docs.python.org/zh-cn/3.12/tutorial/controlflow.html#default-argument-values)；函数创建、返回内部函数：[复合语句 §8.7](https://docs.python.org/zh-cn/3.12/reference/compound_stmts.html#function-definitions)，闭包绑定：[数据模型 §3.2.8.1](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#user-defined-functions)；推导式变量隔离：[表达式 §6.2.4](https://docs.python.org/zh-cn/3.12/reference/expressions.html#displays-for-lists-sets-and-dictionaries)，lambda：[§6.14](https://docs.python.org/zh-cn/3.12/reference/expressions.html#lambda)；常见错误与默认值修正：[UnboundLocalError FAQ](https://docs.python.org/zh-cn/3.12/faq/programming.html#why-am-i-getting-an-unboundlocalerror-when-the-variable-has-a-value)、[循环函数绑定 FAQ](https://docs.python.org/zh-cn/3.12/faq/programming.html#why-do-lambdas-defined-in-a-loop-with-different-values-all-return-the-same-result)；最小异常捕获：[教程 §8.3](https://docs.python.org/zh-cn/3.12/tutorial/errors.html#handling-exceptions)；高阶函数定义与递归示例：[functools 开篇](https://docs.python.org/zh-cn/3.12/library/functools.html)、[阶乘递归关系](https://docs.python.org/zh-cn/3.12/library/functools.html#functools.cache)；递归边界：[RecursionError](https://docs.python.org/zh-cn/3.12/library/exceptions.html#RecursionError)、[当前限制](https://docs.python.org/zh-cn/3.12/library/sys.html#sys.getrecursionlimit)、[设置限制的风险](https://docs.python.org/zh-cn/3.12/library/sys.html#sys.setrecursionlimit)；迭代器消费：[函数式编程指引·迭代器](https://docs.python.org/zh-cn/3.12/howto/functional.html#iterators)，转换与筛选：[map](https://docs.python.org/zh-cn/3.12/library/functions.html#map)、[filter](https://docs.python.org/zh-cn/3.12/library/functions.html#filter)；排序：[sorted](https://docs.python.org/zh-cn/3.12/library/functions.html#sorted)、[键函数](https://docs.python.org/zh-cn/3.12/howto/sorting.html#key-functions)、[稳定性](https://docs.python.org/zh-cn/3.12/howto/sorting.html#sort-stability-and-complex-sorts)、[元组键的字典序比较](https://docs.python.org/zh-cn/3.12/howto/sorting.html#decorate-sort-undecorate)。 |